In [ ]:
# Step 2 — Load Cleaned Dataset

import pandas as pd

print("Loading cleaned dataset...\n")

df = pd.read_csv(
    "data/processed/cleaned_products1.csv"
)

print(df.head())

print("\nDataset Shape:")
print(df.shape)

Loading cleaned dataset...

         asin                                              title  rating  \
0  B08VJFZQ9S  प्लेन कैज़ुअल वियर बेसबॉल कैप पुरुषों और महिला...     0.0   
1  B08VJFYW5Q  यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (काला, फ़्...     0.0   
2  B08VJFYVX9  प्लेन कैज़ुअल वियर बेसबॉल कैप पुरुषों और महिला...     0.0   
3  B08VJFXM7F  यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (सफ़ेद, फ़...     0.0   
4  B08VJFXFTJ  यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (लाल और का...     0.0   

   review_count  price  listPrice                   category  isBestSeller  \
0             0  299.0      499.0  पुरुषों के हैट्स और कैप्स         False   
1             0  299.0      499.0  पुरुषों के हैट्स और कैप्स         False   
2             0  275.0      300.0  पुरुषों के हैट्स और कैप्स         False   
3             0  275.0      300.0  पुरुषों के हैट्स और कैप्स         False   
4             0  275.0      300.0  पुरुषों के हैट्स और कैप्स         False   

   boughtInLastMonth                          

In [2]:
# Step 3 — Import Libraries

import numpy as np

from rank_bm25 import BM25Okapi

from nltk.tokenize import word_tokenize

import nltk

In [3]:
# Step 4 — Download Tokenizer

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gagan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\gagan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
# Step 5 — Tokenize Search Corpus

print("Tokenizing search corpus...\n")

tokenized_corpus = [
    word_tokenize(text.lower())
    for text in df["search_text"]
]

print("Tokenization completed!")

Tokenizing search corpus...

Tokenization completed!


In [5]:
# Step 6 — Build BM25 Index

print("Building BM25 index...\n")

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index created successfully!")

Building BM25 index...

BM25 index created successfully!


In [6]:
# Step 7 — Create Feature Extraction Function

def extract_features(query, product_index):

    # Tokenize query
    tokenized_query = query.lower().split()

    # BM25 score
    bm25_score = bm25.get_scores(
        tokenized_query
    )[product_index]

    # Product row
    product = df.iloc[product_index]

    # Feature dictionary
    features = {

        "bm25_score": bm25_score,

        "rating":
            product["rating"],

        "review_count":
            product["review_count"],

        "price":
            product["price"],

        "is_best_seller":
            int(product["isBestSeller"]),

        "bought_last_month":
            product["boughtInLastMonth"]
    }

    return features

In [7]:
# Step 8 — Test Feature Extraction
features = extract_features(
    "कैप",
    0
)

print(features)


{'bm25_score': np.float64(2.890569328621395), 'rating': np.float64(0.0), 'review_count': np.int64(0), 'price': np.float64(299.0), 'is_best_seller': 0, 'bought_last_month': np.int64(0)}


In [11]:
# for creating training data : Step 1 — Create Training Queries
queries = [
    "कैप",
    "जूते",
    "घड़ी",
    "बैग",
    "टीशर्ट"
]

In [12]:
# Step 2 — Create Empty List

training_data = []

In [13]:
# Step 3 — Generate Training Data

for query in queries:

    # Tokenize query
    tokenized_query = query.lower().split()

    # BM25 scores
    scores = bm25.get_scores(tokenized_query)

    # Top 20 products
    top_n = np.argsort(scores)[::-1][:20]

    for idx in top_n:

        # Extract ranking features
        features = extract_features(
            query,
            idx
        )

        # Synthetic relevance labels
        relevance = 1

        if features["bm25_score"] > 2:
            relevance = 3

        if (
            features["rating"] >= 4 and
            features["review_count"] > 100
        ):
            relevance = 5

        row = {
            "query": query,
            "product_index": idx,
            "relevance": relevance,
            **features
        }

        training_data.append(row)

In [14]:
# Step 4 — Convert To DataFrame

training_df = pd.DataFrame(training_data)
print(training_df.head())

  query  product_index  relevance  bm25_score  rating  review_count  price  \
0   कैप           2084          3    4.233124     0.0             0    0.0   
1   कैप           2794          3    4.218292     0.0             0  475.0   
2   कैप           2125          3    4.201487     0.0             0    0.0   
3   कैप           2118          3    4.201487     0.0             0    0.0   
4   कैप           2577          3    4.149776     3.4             3  279.0   

   is_best_seller  bought_last_month  
0               0                  0  
1               0                  0  
2               0                  0  
3               0                  0  
4               0                  0  


In [ ]:
# Step 5 — Save Training Dataset

training_df.to_csv(
    "data/processed/training_data.csv",
    index=False
)
print("Training dataset saved successfully!")

Training dataset saved successfully!
